# 00 · Data setup and check

Run this first. It shows which source datasets are on this machine and verifies them. Raw data never goes into Git: `Raw_Data/` and `Processed_Data/` are ignored.

| Notebook | Needs |
|---|---|
| 01 Pump vibration | `Raw_Data/uottawa_bearing/` (48 MAT files) |
| 02 Pump event camera | `Raw_Data/xjtu_dv_pump/` (two XJTU-DV-Pump zips, 15.4 GB) |
| 04 Belt Conveyor | `Raw_Data/cora_gear_wear/` (16 MAT) and `Processed_Data/cora_gear_wear/t1_bounded_samples.npz` |
| 05 UCI hydraulic | `Raw_Data/uci_hydraulic/condition_monitoring_of_hydraulic_systems.zip` |
| 06 ESPset | `Raw_Data/espset_v3/features.zip` |
| 07 PHM rock drill | `Raw_Data/phm2022_rock_drill/training_data.zip` |
| 08 Original pump XLSX | `Raw_Data/pump_bearing/{healthy,inner,outer,ball}.xlsx` |

Download routes are in [`Documentation/capstone_details.md`](../Documentation/capstone_details.md). The intake scripts in `Preprocessing/` fetch the Ottawa and CORA files with checksum checks.

In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
ROOT = next(p for p in (here, *here.parents) if (p / "Models").is_dir() and (p / "Preprocessing").is_dir())
RAW, PROCESSED, RESULTS = ROOT / "Raw_Data", ROOT / "Processed_Data", ROOT / "Results"

def summary(path, pattern):
    files = [p for p in path.rglob(pattern) if p.is_file()] if path.is_dir() else ([path] if path.is_file() else [])
    return len(files), round(sum(p.stat().st_size for p in files) / 1e6, 1)

checks = [
    ("01 Pump vibration", RAW / "uottawa_bearing", "*.mat", 48),
    ("02 Pump event camera", RAW / "xjtu_dv_pump", "*.zip", 2),
    ("04 Belt Conveyor infrared", RAW / "cora_gear_wear", "*.mat", 16),
    ("04 Belt Conveyor T1 samples", PROCESSED / "cora_gear_wear" / "t1_bounded_samples.npz", "*", 1),
    ("05 UCI hydraulic", RAW / "uci_hydraulic", "*.zip", 1),
    ("06 ESPset V3", RAW / "espset_v3", "*.zip", 1),
    ("07 PHM rock drill", RAW / "phm2022_rock_drill", "*.zip", 1),
    ("08 Original pump XLSX", RAW / "pump_bearing", "*.xlsx", 4),
]
rows = []
for label, path, pattern, expected in checks:
    n, mb = summary(path, pattern)
    rows.append({"needed by": label, "files": f"{n}/{expected}", "MB": mb, "ready": n >= expected})
table = pd.DataFrame(rows).set_index("needed by")
display(table)
print("All notebooks can run offline." if table["ready"].all() else "Some notebooks will stop until their data is downloaded.")

,files,MB,ready
needed by,,,
01 Pump vibration,48/48,631.5,True
02 Pump event camera,2/2,16488.7,True
04 Belt Conveyor infrared,16/16,390.5,True
04 Belt Conveyor T1 samples,1/1,1.7,True
05 UCI hydraulic,1/1,76.6,True
06 ESPset V3,1/1,0.3,True
07 PHM rock drill,1/1,235.4,True
08 Original pump XLSX,4/4,3.1,True


All notebooks can run offline.


## Verify integrity

Checks the Ottawa files against the publisher SHA-256 values recorded in `Results/pump_uottawa_intake.json`, and the two XJTU zips against Zenodo's published MD5. The zips are 15 GB, so set `VERIFY_LARGE = False` to skip them.

In [2]:
VERIFY_LARGE = True

def digest(path, algorithm):
    h = hashlib.new(algorithm)
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

if (RAW / "uottawa_bearing").is_dir():
    records = json.loads((RESULTS / "pump_uottawa_intake.json").read_text(encoding="utf-8"))["files"]
    bad = [r["name"] for r in records if digest(RAW / "uottawa_bearing" / r["name"], "sha256") != r["source_sha256"]]
    print(f"Ottawa vibration files: {len(records)} checked, {len(bad)} mismatches {bad[:3]}")

if VERIFY_LARGE and (RAW / "xjtu_dv_pump").is_dir():
    for name, expected in {"XJTU-DV-Pump_part1.zip": "c6e192a190011b1bcf93166ead0f2237",
                           "XJTU-DV-Pump_part2.zip": "7684e8f1068cfb26ce48aca41d3619ef"}.items():
        path = RAW / "xjtu_dv_pump" / name
        if path.is_file():
            print(f"{name}: {'MATCH' if digest(path, 'md5') == expected else 'MISMATCH'} with Zenodo MD5")

Ottawa vibration files: 48 checked, 0 mismatches []


XJTU-DV-Pump_part1.zip: MATCH with Zenodo MD5


XJTU-DV-Pump_part2.zip: MATCH with Zenodo MD5


## Fetch missing data (optional, needs internet)

Set `RUN_DOWNLOADS = True` to run the project's checksum-verifying intake scripts for the Ottawa and CORA data. Other sources are downloaded from the links in [`Documentation/capstone_details.md`](../Documentation/capstone_details.md). `Preprocessing/cora_manifest.py` and `Preprocessing/cora_preprocessing.py` are inspection helpers that take a local publisher file as an argument; see their docstrings.

In [3]:
import runpy
import sys

RUN_DOWNLOADS = False

def run(relative_path, *args):
    saved, sys.argv = sys.argv, [relative_path, *args]
    try:
        runpy.run_path(str(ROOT / relative_path), run_name="__main__")
    finally:
        sys.argv = saved

if RUN_DOWNLOADS:
    run("Preprocessing/pump_bearing_independent_intake.py")
    run("Preprocessing/cora_gear_intake.py")
else:
    print("Downloads skipped (RUN_DOWNLOADS = False).")

Downloads skipped (RUN_DOWNLOADS = False).
